### CSV And EXcel files- Structured Data

In [30]:
import os
import pandas as pd

In [31]:
os.makedirs("data/structured_files", exist_ok=True)

In [32]:
# Create sample data
data = {
    'Product': ['Laptop', 'Mouse', 'Keyboard', 'Monitor', 'Webcam'],
    'Category': ['Electronics', 'Accessories', 'Accessories', 'Electronics', 'Electronics'],
    'Price': [999.99, 29.99, 79.99, 299.99, 89.99],
    'Stock': [50, 200, 150, 75, 100],
    'Description': [
        'High-performance laptop with 16GB RAM and 512GB SSD',
        'Wireless optical mouse with ergonomic design',
        'Mechanical keyboard with RGB backlighting',
        '27-inch 4K monitor with HDR support',
        '1080p webcam with noise cancellation'
    ]
}


In [33]:
df = pd.DataFrame(data)
df.to_csv('data/structured_files/products.csv', index=False)

In [34]:
writer = pd.ExcelWriter("data/structured_files/inventory.xlsx")

df.to_excel(writer, sheet_name="Products", index=False)

# Add another sheet

summary_data = {
    "Category": ["Electronics", "Accessories"],
    "Total_Items": [5, 4],
    "Total_Value": [1598.67, 1556.92]
}

pd.DataFrame(summary_data).to_excel(writer, sheet_name="Summary", index=False)

writer.close()

## CSV Processing

#### 📊 CSV Processing Strategies

1. Row-based (CSVLoader):
- ✅ Simple one-row-one-document
- ✅ Good for record lookups
- ❌ Loses table context

1. Intelligent Processing:
- ✅ Preserves relationships
- ✅ Creates summaries
- ✅ Rich metadata
- ✅ Better for Q&A

In [35]:
from langchain_community.document_loaders import CSVLoader

In [36]:
csv_loader = CSVLoader(
    file_path='data/structured_files/products.csv',
    encoding='utf-8',
    csv_args={
        'delimiter': ',',
        'quotechar': '"',
    }
)
csv_loader

In [37]:
csv_loader.load()

[Document(metadata={'source': 'data/structured_files/products.csv', 'row': 0}, page_content='Product: Laptop\nCategory: Electronics\nPrice: 999.99\nStock: 50\nDescription: High-performance laptop with 16GB RAM and 512GB SSD'),
 Document(metadata={'source': 'data/structured_files/products.csv', 'row': 1}, page_content='Product: Mouse\nCategory: Accessories\nPrice: 29.99\nStock: 200\nDescription: Wireless optical mouse with ergonomic design'),
 Document(metadata={'source': 'data/structured_files/products.csv', 'row': 2}, page_content='Product: Keyboard\nCategory: Accessories\nPrice: 79.99\nStock: 150\nDescription: Mechanical keyboard with RGB backlighting'),
 Document(metadata={'source': 'data/structured_files/products.csv', 'row': 3}, page_content='Product: Monitor\nCategory: Electronics\nPrice: 299.99\nStock: 75\nDescription: 27-inch 4K monitor with HDR support'),
 Document(metadata={'source': 'data/structured_files/products.csv', 'row': 4}, page_content='Product: Webcam\nCategory: Ele

In [38]:
from typing import List
from langchain_core.documents import Document

# Method 2: Custom CSV processing for better control
print("\n2️⃣ Custom CSV Processing")
def process_csv_intelligently(filepath: str) -> List[Document]:
    """Process CSV with intelligent document creation"""
    df = pd.read_csv(filepath)
    documents = []
    
    # Strategy 1: One document per row with structured content
    for idx, row in df.iterrows():
        # Create structured content
        content = f"""Product Information:
        Name: {row['Product']}
        Category: {row['Category']}
        Price: ${row['Price']}
        Stock: {row['Stock']} units
        Description: {row['Description']}"""
        
        # Create document with rich metadata
        doc = Document(
            page_content=content,
            metadata={
                'source': filepath,
                'row_index': idx,
                'product_name': row['Product'],
                'category': row['Category'],
                'price': row['Price'],
                'data_type': 'product_info'
            }
        )
        documents.append(doc)
    return documents


2️⃣ Custom CSV Processing


In [39]:
process_csv_intelligently('data/structured_files/products.csv')

[Document(metadata={'source': 'data/structured_files/products.csv', 'row_index': 0, 'product_name': 'Laptop', 'category': 'Electronics', 'price': 999.99, 'data_type': 'product_info'}, page_content='Product Information:\n        Name: Laptop\n        Category: Electronics\n        Price: $999.99\n        Stock: 50 units\n        Description: High-performance laptop with 16GB RAM and 512GB SSD'),
 Document(metadata={'source': 'data/structured_files/products.csv', 'row_index': 1, 'product_name': 'Mouse', 'category': 'Accessories', 'price': 29.99, 'data_type': 'product_info'}, page_content='Product Information:\n        Name: Mouse\n        Category: Accessories\n        Price: $29.99\n        Stock: 200 units\n        Description: Wireless optical mouse with ergonomic design'),
 Document(metadata={'source': 'data/structured_files/products.csv', 'row_index': 2, 'product_name': 'Keyboard', 'category': 'Accessories', 'price': 79.99, 'data_type': 'product_info'}, page_content='Product Informa

### Excel Processing with pandas

In [ ]:
# Process with pandas
excel_file = pd.ExcelFile("data/structured_files/inventory.xlsx")

In [48]:
pd.read_excel("data/structured_files/inventory.xlsx")

,Product,Category,Price,Stock,Description
0,Laptop,Electronics,999.99,50,High-performance laptop with 16GB RAM and 512G...
1,Mouse,Accessories,29.99,200,Wireless optical mouse with ergonomic design
2,Keyboard,Accessories,79.99,150,Mechanical keyboard with RGB backlighting
3,Monitor,Electronics,299.99,75,27-inch 4K monitor with HDR support
4,Webcam,Electronics,89.99,100,1080p webcam with noise cancellation


In [51]:
excel_file.parse(sheet_name="Summary")

,Category,Total_Items,Total_Value
0,Electronics,5,1598.67
1,Accessories,4,1556.92


In [61]:
from langchain_core.documents import Document

def intelligently_process_excel(file_path: str):
    excel_file = pd.ExcelFile(file_path)

    documents = []

    for sheet_name in excel_file.sheet_names:
        excel_content = pd.read_excel(file_path, sheet_name=sheet_name)

        page_content = f"Sheet: {sheet_name}"
        page_content += f" Columns: {", ".join(excel_content.columns)}"
        page_content += f" Rows: {len(excel_content)} "
        page_content += excel_content.to_string(index=False)

        page_content = " ".join(page_content.split())

        doc = Document(
            page_content=page_content,
             metadata={
                'source': file_path,
                'sheet_name': sheet_name,
                'num_rows': len(excel_content),
                'num_columns': len(excel_content.columns),
                'data_type': 'excel_sheet'
            }
        )

        documents.append(doc)

    return documents

        

In [64]:
excel_docs = intelligently_process_excel("data/structured_files/inventory.xlsx")
print(excel_docs)
print(f"Processed {len(excel_docs)} sheets")

[Document(metadata={'source': 'data/structured_files/inventory.xlsx', 'sheet_name': 'Products', 'num_rows': 5, 'num_columns': 5, 'data_type': 'excel_sheet'}, page_content='Sheet: Products Columns: Product, Category, Price, Stock, Description Rows: 5 Product Category Price Stock Description Laptop Electronics 999.99 50 High-performance laptop with 16GB RAM and 512GB SSD Mouse Accessories 29.99 200 Wireless optical mouse with ergonomic design Keyboard Accessories 79.99 150 Mechanical keyboard with RGB backlighting Monitor Electronics 299.99 75 27-inch 4K monitor with HDR support Webcam Electronics 89.99 100 1080p webcam with noise cancellation'), Document(metadata={'source': 'data/structured_files/inventory.xlsx', 'sheet_name': 'Summary', 'num_rows': 2, 'num_columns': 3, 'data_type': 'excel_sheet'}, page_content='Sheet: Summary Columns: Category, Total_Items, Total_Value Rows: 2 Category Total_Items Total_Value Electronics 5 1598.67 Accessories 4 1556.92')]
Processed 2 sheets


### Excel Processing with UnstructuredExcelLoader from langchain_community

- ✅ Handles complex Excel features
- ✅ Preserves formatting info
- ❌ Requires unstructured library

In [65]:
from langchain_community.document_loaders import UnstructuredExcelLoader

In [67]:
# Method 2: UnstructuredExcelLoader
print("\n2️⃣ UnstructuredExcelLoader")
try:
    excel_loader = UnstructuredExcelLoader(
        'data/structured_files/inventory.xlsx',
        mode="elements"
    )
    unstructured_docs = excel_loader.load()
    print(unstructured_docs)

except Exception as e:
    print(f"  ℹ️ Requires unstructured library with Excel support {e}")


2️⃣ UnstructuredExcelLoader
  ℹ️ Requires unstructured library with Excel support No module named 'msoffcrypto'
